# Препроцессинг: ресайз изображений Plant Pathology 2021 → 512px

Запусти этот ноутбук **один раз** (CPU, Internet можно Off). Он уменьшает все train/test
изображения так, чтобы большая сторона была `MAX_SIDE` px, и пересохраняет как JPEG.

После прогона:
1. **Save Version** (Save & Run All) — содержимое `/kaggle/working` станет output'ом.
2. На странице ноутбука: **Output → New Dataset** (или из версии создать датасет).
3. В тренировочном ноутбуке: **Add Data** этот датасет и поставь
   `CFG.DATA_DIR = Path('/kaggle/input/<твой-slug>')`.

Большие JPEG (~4000px) — главный тормоз обучения: их чтение с диска и декод съедают
всё время, GPU простаивает. На 512px файлы в разы меньше → эпохи в разы короче.

In [ ]:
import shutil
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

from PIL import Image
from tqdm.auto import tqdm

SRC_DIR = Path("/kaggle/input/competitions/plant-pathology-2021-fgvc8")
DST_DIR = Path("/kaggle/working/plant-pathology-512")
MAX_SIDE = 512        # большая сторона результата
JPEG_QUALITY = 90
NUM_WORKERS = 4       # на Kaggle обычно 4 vCPU

SUBDIRS = ["train_images", "test_images"]
EXTS = {".jpg", ".jpeg", ".png"}

In [ ]:
def resize_one(args):
    src_path, dst_path = args
    try:
        image = Image.open(src_path)
        # draft ускоряет сам декод исходного большого JPEG
        image.draft("RGB", (MAX_SIDE, MAX_SIDE))
        image = image.convert("RGB")
        w, h = image.size
        scale = MAX_SIDE / max(w, h)
        if scale < 1.0:
            image = image.resize((round(w * scale), round(h * scale)), Image.BILINEAR)
        image.save(dst_path, format="JPEG", quality=JPEG_QUALITY)
        return True
    except Exception as exc:
        print(f"FAILED {src_path}: {exc}")
        return False


def build_jobs():
    jobs = []
    for sub in SUBDIRS:
        src_sub = SRC_DIR / sub
        dst_sub = DST_DIR / sub
        dst_sub.mkdir(parents=True, exist_ok=True)
        for p in src_sub.iterdir():
            if p.suffix.lower() in EXTS:
                jobs.append((p, dst_sub / (p.stem + ".jpg")))
    return jobs

In [ ]:
DST_DIR.mkdir(parents=True, exist_ok=True)

# CSV копируем как есть
for name in ["train.csv", "sample_submission.csv"]:
    src = SRC_DIR / name
    if src.exists():
        shutil.copy(src, DST_DIR / name)
        print(f"copied {name}")

jobs = build_jobs()
print(f"Всего изображений: {len(jobs)}")

ok = 0
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as ex:
    for res in tqdm(ex.map(resize_one, jobs, chunksize=16), total=len(jobs)):
        ok += int(res)

print(f"Готово: {ok}/{len(jobs)} сохранено в {DST_DIR}")

In [ ]:
# Быстрая проверка результата
for sub in SUBDIRS:
    files = list((DST_DIR / sub).glob("*.jpg"))
    print(f"{sub}: {len(files)} файлов")
    if files:
        sample = Image.open(files[0])
        size_kb = files[0].stat().st_size / 1024
        print(f"  пример: {files[0].name} | {sample.size} | {size_kb:.0f} KB")